In [5]:
# =====================================================================
# Tugas P2 - Representasi Data & Pra-pemrosesan
# MK. Pembelajaran Mesin
# Dataset : Loan Approval Prediction Dataset (Kaggle)
# Target  : loan_status (Approved / Rejected)  -> klasifikasi biner
# =====================================================================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, classification_report

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)

PATH = "https://raw.githubusercontent.com/OnyxSal/ML-Loan-Classification/refs/heads/main/loan_approval_dataset.csv"

# ---------------------------------------------------------------------
# 1. Muat data & info kolom
# ---------------------------------------------------------------------
df = pd.read_csv(PATH)
print("=== 1. INFO AWAL ===")
print("Ukuran data:", df.shape)
print("Nama kolom mentah:", list(df.columns))   # ada spasi di depan nama kolom
df.info()

# Rapikan nama kolom dan isi teks (spasi berlebih)
df.columns = df.columns.str.strip()
for c in df.select_dtypes(include=["object", "string"]).columns:
    df[c] = df[c].str.strip()

# Koreksi tipe: loan_id hanya ID, no_of_dependents & loan_term = bilangan bulat diskrit
df = df.set_index("loan_id")
print("\nTipe kolom setelah dirapikan:\n", df.dtypes)

# ---------------------------------------------------------------------
# 2. Missing value
# ---------------------------------------------------------------------
print("\n=== 2. MISSING VALUE ===")
print(df.isna().sum())
# jaga-jaga: string kosong / 'N/A' dianggap missing
df = df.replace(["", "N/A", "n/a", "NA", "nan"], np.nan)
if df.isna().sum().sum() > 0:
    for c in df.columns:
        if df[c].dtype.kind in "if":
            df[c] = df[c].fillna(df[c].median())
        else:
            df[c] = df[c].fillna(df[c].mode()[0])
print("Total missing setelah penanganan:", df.isna().sum().sum())

# ---------------------------------------------------------------------
# 3. Duplikasi
# ---------------------------------------------------------------------
print("\n=== 3. DUPLIKASI ===")
n_dup = df.duplicated().sum()
print("Jumlah baris duplikat:", n_dup)
df = df.drop_duplicates()

# ---------------------------------------------------------------------
# 4. Kategori (ejaan / kapitalisasi) & encoding
# ---------------------------------------------------------------------
print("\n=== 4. KATEGORI ===")
for c in ["education", "self_employed", "loan_status"]:
    print(c, "->", df[c].value_counts().to_dict())

df["education"] = df["education"].map({"Not Graduate": 0, "Graduate": 1})  # ordinal
df["self_employed"] = df["self_employed"].map({"No": 0, "Yes": 1})         # biner
df["loan_status"] = df["loan_status"].map({"Rejected": 0, "Approved": 1})  # target

# ---------------------------------------------------------------------
# 5. Ketidakseimbangan kelas
# ---------------------------------------------------------------------
print("\n=== 5. KESEIMBANGAN KELAS ===")
dist = df["loan_status"].value_counts()
print(dist.rename({1: "Approved", 0: "Rejected"}))
print((dist / len(df) * 100).round(2).rename({1: "Approved %", 0: "Rejected %"}))

# ---------------------------------------------------------------------
# 6. Indikasi kebocoran data (leakage)
# ---------------------------------------------------------------------
print("\n=== 6. CEK LEAKAGE ===")
corr = df.corr()["loan_status"].drop("loan_status").sort_values(key=abs, ascending=False)
print("Korelasi fitur dengan target:\n", corr.round(3))
print("-> loan_id sudah dikeluarkan dari fitur (hanya identitas, bukan prediktor).")
print("-> cibil_score berkorelasi sangat kuat; wajar (skor kredit), tetapi perlu diwaspadai"
      " bila skor ini dihitung setelah keputusan pinjaman.")

# ---------------------------------------------------------------------
# 7. Split data (sebelum outlier & scaling agar tidak bocor)
# ---------------------------------------------------------------------
X = df.drop(columns="loan_status")
y = df["loan_status"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print("\n=== 7. SPLIT 80:20 (stratified) ===")
print("Train:", X_train.shape, "| Test:", X_test.shape)

# ---------------------------------------------------------------------
# 8. Outlier (IQR) -> winsorizing, batas dihitung dari data latih saja
# ---------------------------------------------------------------------
print("\n=== 8. OUTLIER (IQR, data latih) ===")
num_cols = ["no_of_dependents", "income_annum", "loan_amount", "loan_term", "cibil_score",
            "residential_assets_value", "commercial_assets_value",
            "luxury_assets_value", "bank_asset_value"]
bounds, rows = {}, []
for c in num_cols:
    q1, q3 = X_train[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    bounds[c] = (lo, hi)
    n_out = ((X_train[c] < lo) | (X_train[c] > hi)).sum()
    rows.append((c, q1, q3, iqr, lo, hi, n_out))
print(pd.DataFrame(rows, columns=["kolom", "Q1", "Q3", "IQR", "batas_bawah",
                                  "batas_atas", "jml_outlier"]).round(1).to_string(index=False))
for c in num_cols:
    lo, hi = bounds[c]
    X_train[c] = X_train[c].clip(lo, hi)
    X_test[c] = X_test[c].clip(lo, hi)

# ---------------------------------------------------------------------
# 9. Standarisasi (fit pada data latih saja)
# ---------------------------------------------------------------------
print("\n=== 9. STANDARISASI (Z-score) ===")
scaler = StandardScaler()
X_train_s = X_train.copy()
X_test_s = X_test.copy()
X_train_s[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_s[num_cols] = scaler.transform(X_test[num_cols])
print("Rata-rata data latih (≈0):\n", X_train_s[num_cols].mean().round(3).to_dict())
print("Std data latih (≈1):\n", X_train_s[num_cols].std().round(3).to_dict())

# ---------------------------------------------------------------------
# 10. Tampilkan 5 teratas & 5 terbawah data hasil pemrosesan
# ---------------------------------------------------------------------
hasil = pd.concat([X_train_s.assign(loan_status=y_train),
                   X_test_s.assign(loan_status=y_test)]).sort_index()
print("\n=== 10. DATA HASIL PEMROSESAN ===")
print("Ukuran:", hasil.shape)
print("\n5 Teratas:")
print(hasil.head().round(3))
print("\n5 Terbawah:")
print(hasil.tail().round(3))
hasil.to_csv("loan_approval_preprocessed.csv")

# ---------------------------------------------------------------------
# 11. Klasifikasi dengan SVM (NIM genap)
# ---------------------------------------------------------------------
print("\n=== 11. KLASIFIKASI SVM ===")
model = SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced")
model.fit(X_train_s, y_train)
pred = model.predict(X_test_s)
print(f"Akurasi={accuracy_score(y_test, pred):.4f} | F1={f1_score(y_test, pred):.4f}")
print(classification_report(y_test, pred, target_names=["Rejected", "Approved"]))

=== 1. INFO AWAL ===
Ukuran data: (4269, 13)
Nama kolom mentah: ['loan_id', ' no_of_dependents', ' education', ' self_employed', ' income_annum', ' loan_amount', ' loan_term', ' cibil_score', ' residential_assets_value', ' commercial_assets_value', ' luxury_assets_value', ' bank_asset_value', ' loan_status']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4269 entries, 0 to 4268
Data columns (total 13 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   loan_id                    4269 non-null   int64 
 1    no_of_dependents          4269 non-null   int64 
 2    education                 4269 non-null   object
 3    self_employed             4269 non-null   object
 4    income_annum              4269 non-null   int64 
 5    loan_amount               4269 non-null   int64 
 6    loan_term                 4269 non-null   int64 
 7    cibil_score               4269 non-null   int64 
 8    residential_assets_value  4